# Day06 — inference pipeline и сохранение модели

## D6-01 — frozen inputs и notebook setup

**Цель:** подготовить проверенный read-only reference для будущего inference bundle. Входы: Day02 processed/provenance, Day03 config/manifest, Day04 selection/config/validation reference и Day05 frozen protocol/final cache. Выход: типизированный `InferenceReference` с train/validation таблицами и frozen contract.

**Теория.** Config описывает алгоритм и параметры, а fitted estimator хранит выученные vocabulary/IDF и веса classifier. Сохранённые predictions — ответы на конкретные строки; они не позволяют классифицировать новый текст. Поэтому Day06 позже явно восстановит модель на прежнем train, проверит validation replay и сохранит fitted Pipeline. Здесь модели ещё нет.

Поток будущего inference: **raw texts `(N,)` → `preprocess_text` (`minimal-v1`) → clean texts `(N,)` → fitted TF-IDF `(N, V)` sparse → classifier → predictions `(N,)`**. Vocabulary/IDF fit только на train. Ни validation, ни test не добавляются к fit. Test не используется для новых model calls.

Сейчас loaders проверяют hashes, схемы, полные параметры, выбор по CV, версии, IDs/labels/order и непересечение groups. Read-only final-cache reader сверяет сохранённые Day05 outputs; нового fit, predict, CV, download или запуска final evaluation нет. Reference отдаёт только train/validation, без test texts/predictions.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src/finnews_sentiment").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src/finnews_sentiment").is_dir():
    raise RuntimeError("Run from the project root or notebooks directory")
sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.models.inference_artifact import (
    InferenceReference, load_inference_reference,
)

reference: InferenceReference = load_inference_reference(ROOT)
contract = reference.contract
print("Candidate:", contract.candidate_id)
print("Fit policy:", contract.fit_policy)
print("Preprocessing:", contract.preprocessing_policy)
print("Class order:", contract.class_order)
print("Label mapping:", contract.label_mapping)
print("Seed:", contract.random_state)
print("Schema for future bundle:", contract.schema_version)
print("Full effective parameters:", contract.pipeline_parameters)

Candidate: linear_svc
Fit policy: day03_train_only
Preprocessing: minimal-v1
Class order: ('negative', 'neutral', 'positive')
Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Seed: 42
Schema for future bundle: day06-inference-v1
Full effective parameters: {'tfidf': {'analyzer': 'word', 'binary': False, 'decode_error': 'strict', 'dtype': 'float64', 'encoding': 'utf-8', 'input': 'content', 'lowercase': False, 'max_df': 1.0, 'max_features': 5000, 'min_df': 1, 'ngram_range': [1, 2], 'norm': 'l2', 'preprocessor': None, 'smooth_idf': True, 'stop_words': None, 'strip_accents': None, 'sublinear_tf': False, 'token_pattern': '(?u)\\b\\w\\w+\\b', 'tokenizer': None, 'use_idf': True, 'vocabulary': None}, 'classifier': {'C': 1.0, 'class_weight': None, 'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1, 'loss': 'squared_hinge', 'max_iter': 5000, 'multi_class': 'ovr', 'penalty': 'l2', 'random_state': 42, 'tol': 0.0001, 'verbose': 0}}


In [2]:
print("Train DataFrame:", reference.train.shape)
print("Validation DataFrame:", reference.validation.shape)
print("Validation predictions reference:", reference.validation_predictions.shape)
print("Day05 recorded feature count:", reference.day05_n_features)
print("Day05 cached replay:", reference.day05_validation_replay)
display(reference.split_summary)
display(reference.train.head(3))
display(reference.validation_predictions.head(3))

Train DataFrame: (2210, 5)
Validation DataFrame: (553, 5)
Validation predictions reference: (553, 3)
Day05 recorded feature count: 5000
Day05 cached replay: exact_match


,rows,groups,negative,neutral,positive
split,,,,,
train,2210,2206,269,1374,567
validation,553,552,67,344,142
test,690,690,84,428,178


,source_row_id,text,sentiment,text_clean,label_id
1,1,With the new production plant the company woul...,positive,with the new production plant the company woul...,2
4,4,Operating profit rose to EUR 13.1 mn from EUR ...,positive,operating profit rose to eur 13.1 mn from eur ...,2
5,5,"Operating profit totalled EUR 21.1 mn , up fro...",positive,"operating profit totalled eur 21.1 mn , up fro...",2


,source_row_id,true_label_id,pred_label_id
0,2,2,2
1,3,2,2
2,11,2,2


### Источники и проверенные contracts

Таблицы train/validation имеют пять columns: `source_row_id`, `text`, `sentiment`, `text_clean`, `label_id`. Prediction reference содержит три columns: `source_row_id`, `true_label_id`, `pred_label_id`. Их порядок совпадает с validation manifest. Split summary показывает только counts по splits, без test texts.

Каждая строка сохраняет вес в обучении; same-label duplicates остаются в одном split. Непересечение exact-text groups не доказывает независимость похожих предложений. SHA-256 защищает согласованность bytes, semantic checks — смысл схем/IDs/параметров. Implementation hash preprocessing фиксируется **сейчас** для будущего bundle: он не доказывает, какие bytes исходника использовались в утраченной Day05 модели.

In [3]:
sources = pd.DataFrame(
    [{"path": path, "sha256": checksum}
     for path, checksum in sorted(contract.input_checksums.items())]
)
print("Verified input/final files:", len(sources))
display(sources)
print("Preprocessing implementation SHA-256:", contract.preprocessing_sha256)
print("Frozen runtime versions:", contract.runtime_versions)

assert reference.train["source_row_id"].tolist() == list(contract.train_source_row_ids)
assert reference.validation["source_row_id"].tolist() == list(contract.validation_source_row_ids)
assert reference.validation_predictions["source_row_id"].tolist() == list(contract.validation_source_row_ids)
assert reference.validation_predictions["true_label_id"].tolist() == reference.validation["label_id"].tolist()
assert set(contract.train_source_row_ids).isdisjoint(contract.validation_source_row_ids)
print("IDs/order/truth aligned; no fitted Day06 model or new predictions created.")

Verified input/final files: 28


,path,sha256
0,configs/dataset.json,fc43038477b9beeb491c505244b7b0a403b8da46f062f6...
1,configs/experiments_day04.json,d9830dd3805252cc523a2fcb6fc22a28638e2c59d31fee...
2,configs/train_baseline.json,8ffeb9230ae0cbffd6c00939694d028370ea5e825eaabf...
3,data/processed/day03/splits.csv,b51d363cac4c00f18d760aa72d3ca6f5272becced6be45...
4,data/processed/financial_phrasebank_75agree.csv,39dda898a281dc3294be081917b0681524e73edb60f009...
5,reports/day02/provenance.json,8d9e039f9ff7a76efb727a3e21ce2f07232aedb7b38600...
6,reports/day03/config.json,90b5abb6d13b209e69b989d052e9da87bef50a9e936cae...
7,reports/day03/metrics.json,727282dc6860f800b0ed05bcb1d17bef7fff007db424c2...
8,reports/day03/provenance.json,73f72d587bff42974d4eed6274d6886f03843d264aaf7e...
9,reports/day03/split_summary.json,8fe5b294800246fa9883def1a9fc615831385ed2c689a2...


Preprocessing implementation SHA-256: a08d569712d86603ee966aa338d931d3dfe805d66c3a9a5f308704414487f4fd
Frozen runtime versions: {'python': '3.14.6', 'pandas': '3.0.6', 'numpy': '2.5.3', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'matplotlib': '3.11.2', 'joblib': '1.6.0', 'threadpoolctl': '3.7.0'}
IDs/order/truth aligned; no fitted Day06 model or new predictions created.


### Интерпретация и самопроверка

На реальных локальных artifacts: train **2210 строк / 2206 groups**, validation **553 строки / 552 groups**. Day05 записал `V=5000` и `exact_match` validation replay. Это историческая metadata проверенного cache; **в этом section новый Day06 replay не выполняется**, estimator здесь не создаётся. Раздел D6-02 ниже отдельно выполняет train-only build и fresh validation replay; сохранение модели относится к D6-03.

`FrozenInferenceContract` описывает проверенную конфигурацию и provenance. В разделе D6-02 ниже читаются сохранённые `BundleMetadata` и fitted `InferenceBundle`, созданные отдельной build-командой. Операции build/save/load реализованы в `inference_artifact.py`; notebook использует только read-only load. Неизвестный или повреждённый reference вызывает ошибку с путём, а не скрытую подготовку данных или переобучение.

1. Почему saved predictions и config не заменяют fitted estimator?
2. Почему test texts не передаются будущему builder, хотя проверяются hashes final cache?
3. Чем cached Day05 replay отличается от будущего fresh Day06 replay?
4. Какую утечку создаст fit TF-IDF на train + validation?
5. Почему одинаковая длина таблиц ещё не доказывает alignment по source IDs?

## D6-02 — train-only build и точный validation replay

**Цель:** восстановить выбранный по train-only CV алгоритм на прежних train строках. Вход — проверенный `InferenceReference`; выход — `InferenceBundle` с fitted Pipeline **в памяти**, fresh replay evidence и размерами обучения. Запись joblib/manifest относится к D6-03.

**Теория.** Сначала заново проверяются frozen artifacts и неизменность переданного reference, затем `preprocess_text(raw) == text_clean` на всех train/validation строках. Прежний builder создаёт candidate из selection; его полные параметры, включая defaults, сверяются с frozen protocol. TF-IDF vocabulary/IDF и classifier обучаются только на clean train в исходном manifest order, с сохранением duplicates.

После fit один вызов `predict` на validation должен точно воспроизвести все Day04 pred IDs; source IDs, true labels и порядок проверены до fit. `ConvergenceWarning`, несовпадение параметров, preprocessing или replay прекращают build. Validation не добавляется к train; test texts не передаются model calls. Vocabulary/IDF/classes не должны изменяться при replay.

**После D6-03 этот checkpoint читает metadata готового bundle без fit.** Первоначальный D6-02 выполнил явный train-only build; теперь build вызывается отдельно командой из `docs/day06.md`. Если artifact отсутствует, notebook завершится с ошибкой, а не запустит обучение.

In [4]:
from finnews_sentiment.models.inference_artifact import (
    BUNDLE_NAME, InferenceBundle, load_bundle,
)

bundle_path = ROOT / "models/day06" / BUNDLE_NAME
built_bundle: InferenceBundle = load_bundle(bundle_path)
build_metadata = built_bundle.metadata
print("Candidate:", build_metadata.contract.candidate_id)
print("Fit policy:", build_metadata.contract.fit_policy)
print("Train rows:", build_metadata.n_train)
print("Validation rows:", build_metadata.n_validation)
print("TF-IDF train dimensions:", (build_metadata.n_train, build_metadata.n_features))
print("Recorded fresh Day06 validation replay:", build_metadata.validation_replay)
print("Day04 validation reference SHA-256:", build_metadata.validation_reference_sha256)
print("Loaded a completed artifact without fit; explicit build command is in docs/day06.md.")

Candidate: linear_svc
Fit policy: day03_train_only
Train rows: 2210
Validation rows: 553
TF-IDF train dimensions: (2210, 5000)
Recorded fresh Day06 validation replay: exact_match
Day04 validation reference SHA-256: 71d74d84970a956d80306899b0928431177ffcfa7498f3f4fe6e275245a2f987
Loaded a completed artifact without fit; explicit build command is in docs/day06.md.


In [5]:
assert build_metadata.n_train == len(reference.train)
assert build_metadata.n_validation == len(reference.validation_predictions)
assert build_metadata.n_features == reference.day05_n_features
assert build_metadata.contract == reference.contract
assert build_metadata.validation_replay == "exact_match"
assert len(built_bundle.pipeline.named_steps["tfidf"].get_feature_names_out()) == build_metadata.n_features
print("Fitted class IDs:", built_bundle.pipeline.classes_)
print("Train duplicate rows retained:", int(reference.train["text_clean"].duplicated().sum()))
display(reference.validation_predictions.head(3))
print("All validation predictions matched this verified reference, not just the displayed rows.")

Fitted class IDs: [0 1 2]
Train duplicate rows retained: 4


,source_row_id,true_label_id,pred_label_id
0,2,2,2
1,3,2,2
2,11,2,2


All validation predictions matched this verified reference, not just the displayed rows.


### Интерпретация и самопроверка D6-02

При отдельной сборке на реальных данных candidate `linear_svc` обучен на **2210 train строках / 2206 groups**; четыре duplicate rows сверх первого сохранены. TF-IDF имеет **5000 features**, sparse train dimensions `(2210, 5000)`, validation predictions — `(553,)`. Все **553** Day04 predictions воспроизведены с точным alignment; `ConvergenceWarning` отсутствует.

Fresh replay подтверждает воспроизведение замороженного алгоритма, параметров и fit policy на этом validation. Он не доказывает идентичность весов утраченного fitted объекта Day05 и не является новой независимой оценкой качества: validation уже использовалась ранее. На test нет нового predict/transform/score; selection и параметры не меняются.

В отличие от cached replay metadata в D6-01, `build_metadata.validation_replay` относится к fresh replay, выполненному при отдельной сборке сохранённого Day06 Pipeline, а не при текущем read-only запуске notebook. Объект загружен готовым save/load API D6-03. Сохранение/roundtrip уже выполнены при сборке; текущий notebook не вызывает fit, save или predict.

1. Что доказывает exact validation replay и почему он не доказывает побитовую идентичность Day05 весов?
2. Почему validation-only vocabulary token не должен появиться в `vocabulary_`?
3. Почему нельзя удалить четыре retained train duplicates при восстановлении модели?
4. Чем этот явный build отличается от будущего inference, который не вызывает fit?
5. Почему `ConvergenceWarning` останавливает восстановление, хотя sklearn уже вернул estimator?

## D6-03 — bundle, manifest и read-only load

**Цель:** сохранить fitted TF-IDF и classifier вместе с frozen metadata и загрузить их без обучения. Build запускается отдельно из корня проекта:

```bash
PYTHONPATH=src venv/bin/python -m finnews_sentiment.models.inference_artifact
```

Первый явный запуск обучает только прежний train и проверяет validation replay; сохранение выполняется после staging roundtrip/replay. Повтор команды сверяет frozen inputs и готовую пару, не вызывает fit/predict и не меняет bytes artifact. Для нового build есть `--output-dir`; partial/corrupt artifacts не удаляются и не заменяются автоматически.

**Теория:** classifier coefficients имеют смысл только вместе с тем vocabulary/IDF, на котором их обучили. Поэтому `sentiment_bundle.joblib` содержит целый fitted Pipeline и typed metadata. `manifest.json` хранит readable metadata, schema, checksum bundle и checksum fitted state. До deserialize проверяются schema/runtime/preprocessing/hash; после — metadata agreement, fitted components/classes/dimensions/effective params/state.

Joblib загружается только из доверенного локального источника; checksum обнаруживает порчу bytes, но не удостоверяет автора pickle. Для standalone load нужны совместимые src/runtime и оба файла; dataset/configs/reports для `load_bundle` не нужны. D6-01 notebook отдельно читает references в учебных целях.

In [6]:
import hashlib
from finnews_sentiment.models.inference_artifact import MANIFEST_NAME

artifact_files = [bundle_path, bundle_path.with_name(MANIFEST_NAME)]
display(pd.DataFrame([
    {"file": file.relative_to(ROOT).as_posix(), "bytes": file.stat().st_size,
     "sha256": hashlib.sha256(file.read_bytes()).hexdigest()}
    for file in artifact_files
]))
second_load: InferenceBundle = load_bundle(bundle_path)
assert second_load.metadata == built_bundle.metadata
assert second_load.metadata.contract == reference.contract
print("Two read-only loads agree; no fit, predict or artifact writes.")
print("Schema:", second_load.metadata.contract.schema_version)
print("Fitted feature count:", second_load.metadata.n_features)
print("Classes:", second_load.pipeline.classes_)

,file,bytes,sha256
0,models/day06/sentiment_bundle.joblib,170102,72d5f018ccd5babf9debb597e799ab02a76f8dc551babd...
1,models/day06/manifest.json,44552,7b155e9fada9f7d4404801e316d52dc0b0242f52c5de17...


Two read-only loads agree; no fit, predict or artifact writes.
Schema: day06-inference-v1
Fitted feature count: 5000
Classes: [0 1 2]


### Интерпретация и самопроверка D6-03

Реальная пара содержит `linear_svc`, frozen `minimal-v1` preprocessing contract, seed/class mapping, `day03_train_only`, 2210 train IDs, runtime/config/input hashes и replay evidence для всех 553 validation строк. Vocabulary/IDF/classes/weights совпадают до/после serialization; staging roundtrip validation replay прошёл без нового test prediction.

Manifest записывается последним; пара файлов не является общей транзакцией. Partial output требует явной диагностики, а для независимой новой сборки нужно указать новый каталог. Побитовая идентичность joblib между независимыми builds не обещается. Inference load проверяет точные версии Python/NumPy/SciPy/sklearn/joblib/threadpoolctl; matplotlib не требуется.

1. Почему сохранение classifier отдельно от vectorizer опасно?
2. Что проверяется до `joblib.load`, а что возможно проверить только после него?
3. Почему checksum не делает чужой pickle доверенным?
4. Почему повтор notebook или build-команды с готовой парой не должен вызывать fit?
5. Чем roundtrip parity отличается от новой оценки качества модели на независимых данных?

Raw-text prediction API остаётся D6-04; здесь показана проверенная загрузка готового Pipeline.

## D6-04 — batch inference для новых raw texts

**Цель:** передать готовому bundle непустой `Sequence[str]` без true labels и получить `list[SentimentPrediction]`. Один текст передаётся как `[text]`. Каждый result содержит raw `text`, `text_clean`, `pred_label_id: int` и `pred_label: str`; порядок и duplicates сохраняются.

**Теория.** API сначала проверяет весь batch, применяет ту же `preprocess_text` (`minimal-v1`), затем делает один fitted `Pipeline.predict`. Bare str/bytes, None/NaN/нестроковые элементы, пустой batch и пустые после очистки строки отвергаются до model calls; строки не удаляются и не превращаются в `"nan"`.

Поток: raw `(N,)` → clean `(N,)` → sparse TF-IDF `(N, V)` → pred IDs `(N,)` → typed records. Vocabulary/IDF/weights остаются прежними; fit/fit_transform/partial_fit не вызываются. `0=negative`, `1=neutral`, `2=positive`; probability/score не входят в API. LinearSVC не поддерживает `predict_proba`.

Ниже небольшие вручную составленные unlabeled inputs для проверки API, а не ground truth или оценка качества. Реальный bundle прежний; dataset не подменяется fixtures. Отдельный полноценный набор demos остаётся D6-07.

In [7]:
from dataclasses import asdict
from finnews_sentiment.models.predict import SentimentPrediction, predict_texts

raw_batch = [
    "The company reported a 12% increase in operating profit.",
    "<b>No</b> increase in revenue; costs rose by €2.5 million.",
    "The company reported a 12% increase in operating profit.",
]
batch_predictions: list[SentimentPrediction] = predict_texts(built_bundle, raw_batch)
display(pd.DataFrame([asdict(item) for item in batch_predictions]))
single_prediction: list[SentimentPrediction] = predict_texts(built_bundle, [raw_batch[0]])
assert single_prediction == [batch_predictions[0]]
assert batch_predictions[0] == batch_predictions[2]
assert [item.text for item in batch_predictions] == raw_batch
print("Raw/clean/prediction shapes:", (len(raw_batch),), (len(batch_predictions),), (len(batch_predictions),))
print("Fitted TF-IDF dimensions:", (len(raw_batch), build_metadata.n_features))
print("One-text result:", single_prediction[0])
print("Original order and duplicate retained; no fit or artifact writes.")

,text,text_clean,pred_label_id,pred_label
0,The company reported a 12% increase in operati...,the company reported a 12% increase in operati...,2,positive
1,<b>No</b> increase in revenue; costs rose by €...,no increase in revenue; costs rose by €2.5 mil...,2,positive
2,The company reported a 12% increase in operati...,the company reported a 12% increase in operati...,2,positive


Raw/clean/prediction shapes: (3,) (3,) (3,)
Fitted TF-IDF dimensions: (3, 5000)
One-text result: SentimentPrediction(text='The company reported a 12% increase in operating profit.', text_clean='the company reported a 12% increase in operating profit.', pred_label_id=2, pred_label='positive')
Original order and duplicate retained; no fit or artifact writes.


In [8]:
try:
    predict_texts(built_bundle, [raw_batch[0], "<b></b>"])
except ValueError as error:
    print("Whole batch rejected before predict:", error)
else:
    raise AssertionError("Empty-after-clean element must reject the batch")

oov_text = "zxqvplmnbvcxqz"
oov_features = built_bundle.pipeline.named_steps["tfidf"].transform([oov_text])
assert oov_features.nnz == 0
oov_result: SentimentPrediction = predict_texts(built_bundle, [oov_text])[0]
print("OOV sparse feature shape:", oov_features.shape, "; nonzero features:", oov_features.nnz)
print("OOV prediction:", oov_result)
print("A zero feature row is accepted; it does not trigger vocabulary fitting.")

Whole batch rejected before predict: texts[1] is empty after preprocessing
OOV sparse feature shape: (1, 5000) ; nonzero features: 0
OOV prediction: SentimentPrediction(text='zxqvplmnbvcxqz', text_clean='zxqvplmnbvcxqz', pred_label_id=1, pred_label='neutral')
A zero feature row is accepted; it does not trigger vocabulary fitting.


### Интерпретация и самопроверка D6-04

Показаны фактические ответы неизменённого финансового classifier. Unlabeled examples не дают accuracy/F1 и не подтверждают live quality. HTML/whitespace/case нормализуются общей policy; отрицания, числа, валюты и пунктуация сохраняются. Непустая OOV-фраза даёт нулевой TF-IDF row и штатный ответ classifier по его существующим весам/intercept, без новой информации о значении неизвестных слов.

У `SentimentPrediction` четыре объявленных поля; score не выдаётся за probability. Ошибка указывает индекс невалидного элемента. Частичного результата нет даже для валидных строк в том же batch. CSV adapter D6-06 ниже использует этот же API; отдельного расходящегося predictor нет.

1. Что произойдёт, если все слова новой фразы отсутствуют в vocabulary?
2. Почему `[text]` и bare `text` — разные входы API?
3. Почему нельзя вызвать fit_transform на новом batch?
4. Что происходит с двумя одинаковыми строками и с batch, где только последняя строка пуста?
5. Почему несколько правдоподобных ответов без labels не являются новой оценкой macro F1?

## D6-05 — consistency и standalone inference

**Цель:** подтвердить согласованность всей цепочки raw → minimal-v1 → fitted Pipeline → typed prediction и независимость загрузки/предсказания от training artifacts. Используется существующий bundle; fit, новый save, test transform/predict/score и tuning запрещены.

**Теория.** Checksums подтверждают неизменность bytes, но сами по себе не доказывают совпадение preprocessing или предсказаний. Поэтому для **всех** train/validation строк сравниваем `preprocess_text(raw) == text_clean`; для всего validation проверяем IDs, true labels и порядок. Predictions загруженного Pipeline и public raw-text API должны точно совпадать с Day04 reference. Хеш aligned records включает source IDs, true labels и pred IDs — одинакового набора labels недостаточно.

Состояние vocabulary/IDF/classes/weights и metadata сравнивается до/после model calls. Serialization evidence берётся из проверенных manifest/state checksum/build replay D6-03: сейчас модель не сохраняется повторно и новый before-save объект не создаётся.

Для отдельного subprocess копируются только `src` и bundle/manifest; dataset/configs/reports не копируются. Python запускается с `-I -B`, новой cwd, guards на training/network/writes и чтение вне изолированных файлов/runtime. Guard-пробы выполняются до inference; при import запрещено чтение artifact. Это проверка access boundary через Python audit, не OS security sandbox. Новый процесс должен вернуть те же raw/clean/IDs/names для двух новых unlabeled фраз, что и parent API.

Только после всех проверок записывается `reports/day06/consistency.json`. В report нет timestamp, временных путей, PID или self-checksum; при тех же inputs/code/runtime он byte-identical. Ошибка не перезаписывает прошлый report. Это parity evidence, не новая оценка accuracy/F1.

In [9]:
from finnews_sentiment.models.inference_consistency import (
    ConsistencyRun, check_inference_consistency,
)

consistency_run: ConsistencyRun = check_inference_consistency(ROOT)
consistency_report = consistency_run.report
parity = consistency_report["parity"]
print("Report:", consistency_run.report_path.relative_to(ROOT))
print("Candidate:", consistency_report["candidate_id"])
print("Raw/clean rows checked:", parity["n_train"], "train +", parity["n_validation"], "validation")
print("Validation parity:", parity["status"])
print("Aligned reference/Pipeline/API SHA-256:", parity["reference_sha256"])
assert parity["reference_sha256"] == parity["pipeline_sha256"] == parity["public_api_sha256"]
assert consistency_report["state_before_sha256"] == consistency_report["state_after_sha256"]
print("Serialization evidence:", consistency_report["serialization_evidence"])
print("Fitted state unchanged:", consistency_report["state_before_sha256"])
print("Frozen inputs/artifacts:", len(consistency_report["input_checksums"]))
print("Source files:", len(consistency_report["source_checksums"]))

Report: reports/day06/consistency.json
Candidate: linear_svc
Raw/clean rows checked: 2210 train + 553 validation
Validation parity: exact_match
Aligned reference/Pipeline/API SHA-256: 228ab432a817bcfcbd4f0b16c7c854f7c0a394d8ae22bedaa4db0724d431ecb8
Serialization evidence: D6-03 manifest state checksum and recorded exact validation replay
Fitted state unchanged: 7623ab0d4d555aed3c73944f59f85acb23e141ada2794fa826c9d7231383b7f7
Frozen inputs/artifacts: 30
Source files: 26


In [10]:
standalone_evidence = consistency_report["standalone"]
assert standalone_evidence["status"] == "passed"
assert standalone_evidence["import_without_artifact_io"]
assert standalone_evidence["training_guard_checked"]
assert standalone_evidence["file_guard_checked"]
assert standalone_evidence["network_guard_checked"]
print("Standalone:", standalone_evidence["status"])
print("Standalone runtime:", standalone_evidence["runtime_versions"])
display(pd.DataFrame(standalone_evidence["predictions"]))
print("Parent/child predictions and fitted state agree; no dataset/configs/reports were supplied to child.")

Standalone: passed
Standalone runtime: {'python': '3.14.6', 'numpy': '2.5.3', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'joblib': '1.6.0', 'threadpoolctl': '3.7.0'}


,text,text_clean,pred_label_id,pred_label
0,Aster Components announced revenue of €7.4 mil...,aster components announced revenue of €7.4 mil...,1,neutral
1,<b>No</b> change in the company's annual divid...,no change in the company's annual dividend was...,1,neutral


Parent/child predictions and fitted state agree; no dataset/configs/reports were supplied to child.


### Интерпретация и самопроверка D6-05

Проверяются **2210 train + 553 validation** raw/clean pairs и все **553** validation predictions по сохранённому порядку. В main process разрешены loaded-Pipeline validation replay, public API validation replay и новые standalone smoke inputs; в child — только новые smoke inputs. Test не получает новых model calls. Повтор report/notebook не выполняет fit, serialization или независимую оценку качества.

Один kernel может содержать уже созданные переменные, training modules и правильные импорты из истории. Отдельный процесс с изолированными файлами проверяет, что predict требует только совместимый src/runtime и сохранённую пару. Guards и преднамеренно failing import tests подтверждают, что неожиданные чтения/training/network не превращаются в зелёный evidence.

Smoke outputs не имеют ground-truth labels. Exact replay подтверждает воспроизведение frozen решения на уже использованном validation; он не является новым holdout, не доказывает live quality или идентичность утраченных весов Day05.

1. Почему успешный load в текущем kernel не доказывает standalone работу?
2. Почему hash labels без source IDs/order недостаточен для проверки parity?
3. Какие проверки ловят raw/clean skew и изменение vocabulary/IDF/weights?
4. Почему D6-05 не должен заново запускать build/save ради roundtrip?
5. Чем guard проверки доступа отличаются от оценки качества classifier?

## D6-06 — CSV inference adapter и CLI

**Цель:** прочитать новый unlabeled UTF-8 CSV и сохранить исходные columns плюс `text_clean`, `pred_label_id`, `pred_label`. Обязательна только колонка `text`; sentiment/labels/provider/headline не нужны. Adapter вызывает те же `load_bundle` и `predict_texts`, без fit.

**Теория.** CSV читается как строки, без автоматического NA/type inference: literal `NA`/`nan`, leading zeros, quotes, newlines и валюты сохраняются. Header должен быть уникальным, непустым и без output column collisions; число полей каждой строки должно совпадать с header. Ни строки, ни duplicates не удаляются. Весь batch проверяется до predict/output.

Пути — `Path`; input=output, existing output (включая symlinks/hardlinks) запрещены. После inference входные bytes повторно сверяются. Output сначала полностью сериализуется, затем публикуется без overwrite; сбой не оставляет частичный CSV. Новый output сохраняет значения исходных fields и порядок, не byte formatting исходного CSV; без pandas index.

Ниже временные новые inputs для API/CSV equivalence. Real model bundle прежний; real dataset в D6-01/D6-05 не заменяется fixtures. Постоянный demo dataset остаётся D6-07.

In [11]:
import csv
from tempfile import TemporaryDirectory
from finnews_sentiment.models.predict import predict_for_file

csv_input_table = pd.DataFrame({
    "request_id": ["0001", "0002", "0001"],
    "text": raw_batch,
    "note": ["NA", 'quoted "note", €\nnext line', "NA"],
})
with TemporaryDirectory(prefix="day06-csv-notebook-") as directory:
    csv_input_path = Path(directory) / "new_news.csv"
    csv_output_path = Path(directory) / "predictions.csv"
    with csv_input_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(csv_input_table.columns.tolist())
        writer.writerows(csv_input_table.itertuples(index=False, name=None))
    csv_input_before = csv_input_path.read_bytes()
    predict_for_file(bundle_path, csv_input_path, csv_output_path)
    csv_output_table = pd.read_csv(csv_output_path, dtype="str", keep_default_na=False)
    assert csv_input_path.read_bytes() == csv_input_before

pd.testing.assert_frame_equal(csv_output_table[csv_input_table.columns], csv_input_table)
assert csv_output_table["text_clean"].tolist() == [item.text_clean for item in batch_predictions]
assert csv_output_table["pred_label_id"].tolist() == [str(item.pred_label_id) for item in batch_predictions]
assert csv_output_table["pred_label"].tolist() == [item.pred_label for item in batch_predictions]
print("CSV input/output shapes:", csv_input_table.shape, csv_output_table.shape)
display(csv_output_table)
print("All fields/order/duplicates preserved; CSV predictions equal public API.")

CSV input/output shapes: (3, 3) (3, 6)


,request_id,text,note,text_clean,pred_label_id,pred_label
0,0001,The company reported a 12% increase in operati...,NA,the company reported a 12% increase in operati...,2,positive
1,0002,<b>No</b> increase in revenue; costs rose by €...,"quoted ""note"", €\nnext line",no increase in revenue; costs rose by €2.5 mil...,2,positive
2,0001,The company reported a 12% increase in operati...,NA,the company reported a 12% increase in operati...,2,positive


All fields/order/duplicates preserved; CSV predictions equal public API.


In [12]:
with TemporaryDirectory(prefix="day06-csv-errors-") as directory:
    bad_input = Path(directory) / "bad.csv"
    bad_output = Path(directory) / "result.csv"
    bad_input.write_text('text\n"A valid financial sentence."\n"<b></b>"\n', encoding="utf-8")
    try:
        predict_for_file(bundle_path, bad_input, bad_output)
    except ValueError as error:
        print("Invalid batch rejected:", error)
    else:
        raise AssertionError("Empty-after-clean CSV row must fail")
    assert not bad_output.exists()
    bad_output.write_bytes(b"preserve existing output")
    try:
        predict_for_file(bundle_path, bad_input, bad_output)
    except FileExistsError as error:
        print("Existing output protected:", error)
    else:
        raise AssertionError("Existing output must not be replaced")
    assert bad_output.read_bytes() == b"preserve existing output"
print("Input errors did not publish a partial CSV; no model training or artifact writes.")

Invalid batch rejected: texts[1] is empty after preprocessing
Existing output protected: Output already exists: /tmp/day06-csv-errors-k4j65lmh/result.csv
Input errors did not publish a partial CSV; no model training or artifact writes.


### Интерпретация и самопроверка D6-06

Временный CSV `(3, 3)` превратился в `(3, 6)`: три исходных columns плюс три prediction columns. В `request_id` сохранены leading zeros/повторы, в `note` — literal `NA`, comma/quotes/newline; исходный `text` сохранён отдельно от clean text. Все N/order/duplicates и outputs API совпадают.

Из корня CLI вызывается так (выбрать существующий input и новый output):

```bash
PYTHONPATH=src venv/bin/python -m finnews_sentiment.models.predict --model models/day06/sentiment_bundle.joblib --input path/to/new_news.csv --output path/to/predictions.csv
```

CLI выдаёт понятную ошибку (exit 2) без traceback при неверном CSV/model/path и не запускает автоматическую подготовку artifacts. UTF-8 output содержит header, не pandas index. Outputs — labels без probability/score; smoke inputs без ground truth не дают accuracy/F1.

1. Почему inference CSV не обязан иметь колонку sentiment?
2. Почему `NA` не следует автоматически трактовать как missing text?
3. Чем сохранение parsed field values отличается от byte-identical CSV formatting?
4. Что произойдёт с duplicate text и уже существующим output?
5. Почему поздняя невалидная строка должна отклонять весь batch?